In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE, SMOTENC
from imblearn.under_sampling import RandomUnderSampler

df = pd.read_csv("../data/processed/kidney_features.csv")
df.shape

(400, 28)

In [2]:
target_col = "classification"  # adjust to your actual column name
print(df[target_col].value_counts())
print(df[target_col].value_counts(normalize=True))

classification
1    250
0    150
Name: count, dtype: int64
classification
1    0.625
0    0.375
Name: proportion, dtype: float64


In [3]:
X = df.drop(columns=[target_col, "id"])
y = df[target_col]
X.dtypes

age                     float64
bp                      float64
sg                      float64
al                      float64
su                      float64
rbc                       int64
pc                        int64
pcc                       int64
ba                        int64
bgr                     float64
bu                      float64
sc                      float64
sod                     float64
pot                     float64
hemo                    float64
pcv                     float64
wc                      float64
rc                      float64
htn                       int64
dm                        int64
cad                       int64
appet                     int64
pe                        int64
ane                       int64
bun_creatinine_ratio    float64
anemia_ckd_flag           int64
dtype: object

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print("Train:", X_train.shape, y_train.value_counts(normalize=True).to_dict())
print("Test :", X_test.shape,  y_test.value_counts(normalize=True).to_dict())

Train: (320, 26) {1: 0.625, 0: 0.375}
Test : (80, 26) {1: 0.625, 0: 0.375}


In [5]:
y_train.value_counts()

classification
1    200
0    120
Name: count, dtype: int64

In [6]:
sm = SMOTE(random_state=42)
X_train_smote, y_train_smote = sm.fit_resample(X_train, y_train)
print(y_train_smote.value_counts())
print("New train shape:", X_train_smote.shape)

classification
1    200
0    200
Name: count, dtype: int64
New train shape: (400, 26)


In [7]:
rus = RandomUnderSampler(random_state=42)
X_train_under, y_train_under = rus.fit_resample(X_train, y_train)
print(y_train_under.value_counts())
print("New train shape:", X_train_under.shape)

classification
0    120
1    120
Name: count, dtype: int64
New train shape: (240, 26)


In [8]:
# confirm test set was never touched by any fit_resample call
assert len(X_test) == int(round(len(df) * 0.2)), "test set size looks off"
print("X_test index sample:", X_test.index[:5].tolist())
print("Any overlap between train and test indices?",
      bool(set(X_train.index) & set(X_test.index)))

X_test index sample: [342, 204, 233, 366, 120]
Any overlap between train and test indices? False


## Class Imbalance Decision
   Chose SMOTE over undersampling: undersampling shrinks an already-small
   400-row dataset down to 240 training rows, too little to train reliably.
   SMOTE preserves all real data and balances via synthetic minority samples.
   Class weighting will also be tested as a non-resampling alternative in Day 6.